# 12 — Data Module Quickstart

Market data fetching, universe management, provider factory, ticker metadata, and currency detection. See the [data README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/data/README.md) for full documentation.

In [ ]:
from __future__ import annotations
import os
from pathlib import Path

# Ensure CWD is the project root so instrument-master path resolution works.
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "pyproject.toml").exists():
        os.chdir(p)
        break

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

from swing_screener.data.providers import get_market_data_provider
from swing_screener.data import load_universe_from_package, fetch_ohlcv
from swing_screener.data.ticker_info import get_ticker_info
from swing_screener.data.currency import detect_currency

## Provider Factory

Create the active market data provider. Defaults to yfinance when no environment config is set.

In [ ]:
provider = get_market_data_provider()
print(f"Active provider: {type(provider).__name__}")

## Fetch OHLCV

Download daily OHLCV data for a few tickers. The returned DataFrame uses a MultiIndex with `(field, ticker)` columns.

In [ ]:
df = provider.fetch_ohlcv(["AAPL", "MSFT", "SPY"], "2024-06-01", "2024-12-31", use_cache=False)
print(f"Shape: {df.shape}")
df["Close"].tail(5)

## Load a Universe

Load a packaged universe of tickers. Available universes include sector- and index-based snapshots.

In [ ]:
tickers = load_universe_from_package("broad_market_stocks")
print(f"Loaded {len(tickers)} tickers from broad_market_stocks")
print(f"First 10: {tickers[:10]}")

## Ticker Info & Currency Detection

Fetch company metadata (name, sector, currency) from Yahoo Finance, then detect trading currency via suffix heuristics and instrument master lookups.

In [ ]:
info = get_ticker_info("AAPL")
print(f"AAPL info: {info}")
print(f"ASML.AS  → {detect_currency('ASML.AS')}  (expected EUR)")
print(f"AAPL     → {detect_currency('AAPL')}     (expected USD)")
print(f"BP.L     → {detect_currency('BP.L')}     (expected GBP)")